# SDF experiments on Colab
Select an A100 under Runtime → Change runtime type. Uses the existing 575 documents; no API keys or SSH are needed.
Run setup and launch, then rerun the monitor cell while training. Results use unique names and previous files are preserved.


In [ ]:
import pathlib, subprocess, sys
repo = pathlib.Path('/content/sdfqwen')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/colab-experiments', '--single-branch',
                    'https://github.com/Beasleydog/sdfqwen.git', str(repo)], check=True)
print(subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True))
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
assert not (repo / '.env').exists(), 'Unexpected .env in cloned repository'
print('Existing documents:', len(list((repo / 'datagen/outputs').glob('*.txt'))))


In [ ]:
import pathlib, subprocess, sys
repo = pathlib.Path('/content/sdfqwen')
subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
launch = subprocess.run([sys.executable, str(repo / 'experiments/start_colab.py')],capture_output=True,text=True)
print(launch.stdout)
launch.check_returncode()


In [ ]:
from pathlib import Path
import json
repo = Path('/content/sdfqwen')
status = repo / 'colab_status.json'
print(status.read_text() if status.exists() else 'Starting')
log = repo / 'colab_job.log'
if log.exists():
    lines = log.read_text(errors='replace').replace('\r','\n').splitlines()
    progress = [line for line in lines if '/120' in line or '/360' in line]
    events = [line for line in lines if line.startswith(('EVALUATED','COMPLETE',"{'",'Traceback','TypeError','RuntimeError','ImportError'))]
    print('\n'.join(events[-10:] + progress[-1:]))
    if '"state": "failed"' in status.read_text():
        print('\n'.join(lines[-40:]))
print('Evaluation files:', [str(p.relative_to(repo)) for p in (repo/'experiment_results').glob('*/*.json') if p.stem == 'base' or p.stem.startswith('step_')])


In [ ]:
import json
from pathlib import Path
from google.colab import files
repo = Path('/content/sdfqwen')
state = json.loads((repo/'colab_status.json').read_text())
if state['state'] in ('complete', 'failed'):
    files.download(str(repo/'colab_results.zip'))
else:
    print('Job is still running. Rerun this download cell after completion.')
